- orbital area: eye area 
- nose bulge: nose area
- cheek bulge: cheek area
- ear position: ear area
- whisker change: side bart area

# Optional DeepLabCut Crop Workflow

This notebook is an advanced/archival workflow for DeepLabCut-based muzzle crops. The primary public workflow is CLI-first:

```powershell
python scripts/check_setup.py
python scripts/train.py
```

Use the canonical local dataset layout from `dataset/README.md`:

```text
dataset/mouse_dataset/
  MouseGrimaceFaces_main.csv
  MouseGrimaceFaces_mgs.csv
  images_mgs/
  images_perfect/
  muzzle_crops/
```

Reusable filtering and training logic should live in `src/`; this notebook is for exploratory crop generation only.


Question: if mgs is a mix of 0,1,2 and 9, should I keep it or remove it???

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from paths import IMAGES_MGS_DIR, MAIN_CSV, MGS_CSV, MUZZLE_CROPS_DIR

print("Project:", PROJECT_ROOT)
print("MGS images:", IMAGES_MGS_DIR)

In [ ]:
from pathlib import Path
import shutil
import pandas as pd

# -------------------------
# Paths
# -------------------------
DATASET_DIR = Path("mouse_dataset")

IMAGE_DIR = DATASET_DIR / "images"
MGS_CSV = DATASET_DIR / "MouseGrimaceFaces_mgs.csv"
OUTPUT_DIR = DATASET_DIR / "images_mgs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Load MGS csv
# -------------------------
mgs_df = pd.read_csv(MGS_CSV)

print("Original MGS rows:", len(mgs_df))
print(mgs_df.head())

# -------------------------
# Score columns
# -------------------------
# columns like ot1, nb1, cb1, ep1, wc1, ...
score_cols = [
    col for col in mgs_df.columns
    if col not in ["index", "subset"]
]

# Convert scores to numeric:
# valid scores: 0, 1, 2
# invalid: 9
# missing/no score: "-", NaN, empty cells
scores = mgs_df[score_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

# -------------------------
# Filter usable images
# 至少有一个有效分数(0,1,2)，不能出现 9
# -------------------------
# Condition 1: at least one valid score 0/1/2 
has_valid_score = scores.isin([0, 1, 2]).any(axis=1)

# Condition 2: no score equals 9
has_no_9 = ~scores.eq(9).any(axis=1)

usable_df = mgs_df[has_valid_score & has_no_9].copy()

print("Usable MGS rows:", len(usable_df))
print("Removed rows:", len(mgs_df) - len(usable_df))

# -------------------------
# Copy usable images
# -------------------------
copied = 0
missing_files = []

for filename in usable_df["index"]:
    src = IMAGE_DIR / filename
    dst = OUTPUT_DIR / filename

    if src.exists():
        shutil.copy2(src, dst)
        copied += 1
    else:
        missing_files.append(filename)

print("Copied images:", copied)
print("Missing image files:", len(missing_files))

if missing_files:
    print("First missing files:")
    print(missing_files[:10])

# Optional: save list of usable images
usable_csv_path = DATASET_DIR / "usable_mgs_images.csv"
usable_df.to_csv(usable_csv_path, index=False)

print("Saved usable image list to:", usable_csv_path)
print("Copied images stored under:", OUTPUT_DIR)

## DLC Training
Now you need to select images and train DLC.  
There are 2163 usable images.  
I plan to mark 250 images by myself and try out the effect.  
I plan to use those body parts. 
- nose
- eye_left
- eye_right

- conda activate DEEPLABCUT  
- python -m deeplabcut


apply training result on our images

In [ ]:
import deeplabcut

config_path = "mouse-mgs-2026-06-04/config.yaml"

deeplabcut.analyze_time_lapse_frames(
    config_path,
    directory=str(IMAGES_MGS_DIR),
    frametype=".jpg",
    save_as_csv=True,
)

read the h5 file of DLC after the step above

In [ ]:
from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
from tqdm import tqdm

dlc_h5_path = IMAGES_MGS_DIR / "image_predictions_DLC_Resnet50_mouseJun4shuffle1_snapshot_best-200.h5"

image_dir = IMAGES_MGS_DIR
output_dir = MUZZLE_CROPS_DIR
output_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_hdf(dlc_h5_path)

scorer = df.columns.get_level_values("scorer")[0]
individual = df.columns.get_level_values("individuals")[0]

print("scorer:", scorer)
print("individual:", individual)
print("bodyparts:", df.columns.get_level_values("bodyparts").unique().tolist())

In [ ]:
print(df.columns)

In [ ]:
scorer = df.columns.get_level_values(0)[0]

bodyparts = df.columns.get_level_values(1).unique()
print("scorer:", scorer)
print("bodyparts:")
for bp in bodyparts:
    print(bp)

## crop images

must nose, left_eye, right_eye 3 points all > PCUTOFF

In [ ]:
NOSE = "nose-tip"
EYE_L = "left_eye"
EYE_R = "right_eye"

In [ ]:
from PIL import Image
import numpy as np
from tqdm import tqdm
import os

image_dir = IMAGES_MGS_DIR
output_dir = MUZZLE_CROPS_DIR
output_dir.mkdir(parents=True, exist_ok=True)

NOSE = "nose"
EYE_L = "left_eye"
EYE_R = "right_eye"

PCUTOFF = 0.5
BOX_SCALE = 3.0
MIN_BOX_SIZE = 80

def get_point(row, bodypart):
    x = row[(scorer, individual, bodypart, "x")]
    y = row[(scorer, individual, bodypart, "y")]
    p = row[(scorer, individual, bodypart, "likelihood")]
    return float(x), float(y), float(p)

saved = 0
skipped_not_in_dlc = 0
skipped_low_confidence = 0

for image_path in tqdm(sorted(image_dir.glob("*"))):
    if image_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
        continue

    # 关键修改：用完整路径，而不是 image_path.name
    dlc_index = str(image_path)

    if dlc_index not in df.index:
        skipped_not_in_dlc += 1
        continue

    row = df.loc[dlc_index]

    nose_x, nose_y, nose_p = get_point(row, NOSE)
    eye_l_x, eye_l_y, eye_l_p = get_point(row, EYE_L)
    eye_r_x, eye_r_y, eye_r_p = get_point(row, EYE_R)

    if min(nose_p, eye_l_p, eye_r_p) < PCUTOFF:
        skipped_low_confidence += 1
        continue

    img = Image.open(image_path).convert("RGB")
    W, H = img.size

    eye_dist = np.sqrt((eye_l_x - eye_r_x) ** 2 + (eye_l_y - eye_r_y) ** 2)
    box_size = max(eye_dist * BOX_SCALE, MIN_BOX_SIZE)

    center_x = nose_x
    center_y = nose_y

    x1 = int(center_x - box_size / 2)
    x2 = int(center_x + box_size / 2)
    y1 = int(center_y - box_size / 2)
    y2 = int(center_y + box_size / 2)

    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(W, x2)
    y2 = min(H, y2)

    crop = img.crop((x1, y1, x2, y2))
    crop = crop.resize((224, 224))

    crop.save(output_dir / image_path.name)

    saved += 1

print("Saved crops:", saved)
print("Skipped because not in DLC index:", skipped_not_in_dlc)
print("Skipped because low confidence:", skipped_low_confidence)

In [ ]:
import matplotlib.pyplot as plt
import random

crop_paths = list(output_dir.glob("*"))

sample_paths = random.sample(crop_paths, min(25, len(crop_paths)))

plt.figure(figsize=(12, 12))

for i, path in enumerate(sample_paths):
    img = Image.open(path)
    plt.subplot(5, 5, i + 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(path.name[:10])

plt.tight_layout()
plt.show()

nose + at least one eye 

In [ ]:
NOSE = "nose"
EYE_L = "left_eye"
EYE_R = "right_eye"

PCUTOFF_NOSE = 0.5
PCUTOFF_EYE = 0.3
BOX_SIZE = 180

saved = 0
skipped_not_in_dlc = 0
skipped_low_confidence = 0

for image_path in tqdm(sorted(image_dir.glob("*"))):
    if image_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
        continue

    dlc_index = str(image_path)

    if dlc_index not in df.index:
        skipped_not_in_dlc += 1
        continue

    row = df.loc[dlc_index]

    nose_x, nose_y, nose_p = get_point(row, NOSE)
    eye_l_x, eye_l_y, eye_l_p = get_point(row, EYE_L)
    eye_r_x, eye_r_y, eye_r_p = get_point(row, EYE_R)

    # 只要求 nose 可靠
    if nose_p < PCUTOFF_NOSE:
        skipped_low_confidence += 1
        continue

    # 检查 nose 坐标是否正常
    if not np.isfinite([nose_x, nose_y]).all():
        skipped_low_confidence += 1
        continue

    img = Image.open(image_path).convert("RGB")
    W, H = img.size

    if eye_l_p >= PCUTOFF_EYE and eye_r_p >= PCUTOFF_EYE:
        eye_dist = np.sqrt((eye_l_x - eye_r_x) ** 2 + (eye_l_y - eye_r_y) ** 2)
        box_size = max(eye_dist * 3.0, 100)
    else:
        box_size = BOX_SIZE

    if not np.isfinite(box_size) or box_size <= 0:
        skipped_low_confidence += 1
        continue

    center_x = nose_x
    center_y = nose_y

    x1 = int(center_x - box_size / 2)
    x2 = int(center_x + box_size / 2)
    y1 = int(center_y - box_size / 2)
    y2 = int(center_y + box_size / 2)

    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(W, x2)
    y2 = min(H, y2)

    if x2 <= x1 or y2 <= y1:
        skipped_low_confidence += 1
        continue

    crop = img.crop((x1, y1, x2, y2))
    crop = crop.resize((224, 224))
    crop.save(output_dir / image_path.name)
    saved += 1

print("Saved crops:", saved)
print("Skipped because not in DLC index:", skipped_not_in_dlc)
print("Skipped because low confidence:", skipped_low_confidence)

In [ ]:
import matplotlib.pyplot as plt
import random

crop_paths = list(output_dir.glob("*"))

sample_paths = random.sample(crop_paths, min(25, len(crop_paths)))

plt.figure(figsize=(12, 12))

for i, path in enumerate(sample_paths):
    img = Image.open(path)
    plt.subplot(5, 5, i + 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(path.name[:10])

plt.tight_layout()
plt.show()

## crop the area below eyes

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
from tqdm import tqdm

image_dir = IMAGES_MGS_DIR
output_dir = MUZZLE_CROPS_DIR.with_name("muzzle_crops_v3")
output_dir.mkdir(parents=True, exist_ok=True)

h5_files = list(image_dir.glob("*.h5"))
if len(h5_files) == 0:
    raise FileNotFoundError("No DLC .h5 file found")

dlc_h5_path = h5_files[0]
df = pd.read_hdf(dlc_h5_path)

print("Using DLC file:", dlc_h5_path)
print("Index example:", df.index[:3])
print("Column names:", df.columns.names)

# =========================
# DLC MultiIndex
# =========================
scorer = df.columns.get_level_values("scorer")[0]

if "individuals" in df.columns.names:
    individual = df.columns.get_level_values("individuals")[0]
else:
    individual = None

bodyparts = list(df.columns.get_level_values("bodyparts").unique())
print("Available bodyparts:", bodyparts)

# =========================
# Bodypart names
# =========================
NOSE = "nose"
EYE_L = "left_eye"
EYE_R = "right_eye"

# 如果你的 DLC 里有 mouth / whisker，可以自动辅助扩展 crop
OPTIONAL_PARTS = [
    "mouth",
    "mouth_left",
    "mouth_right",
    "upper_lip",
    "lower_lip",
    "muzzle",
    "left_muzzle",
    "right_muzzle",
    "whisker",
    "left_whisker",
    "right_whisker",
    "whisker_left",
    "whisker_right",
]

# =========================
# Crop parameters
# =========================
PCUTOFF_MAIN = 0.5
PCUTOFF_OPTIONAL = 0.3

OUTPUT_SIZE = 224

# crop 大小
MIN_W = 90
MIN_H = 70

FRONT_W_SCALE = 2.6
FRONT_H_SCALE = 1.8

SIDE_W_SCALE = 2.4
SIDE_H_SCALE = 1.6

# 关键参数：
# crop 中心从 nose 往 mouth / whisker 方向移动
NOSE_FORWARD_SHIFT = 0.35

# crop 上边界不要太靠近眼睛
# 越大，越不容易包含眼睛
REMOVE_EYE_MARGIN = 0.25

# crop 向下偏移，适合 front view
DOWN_SHIFT_FRONT = 0.45

# side view 沿 eye -> nose 方向继续向前
FORWARD_SHIFT_SIDE = 0.55


# =========================
# Helper functions
# =========================
def col_key(bodypart, coord):
    if individual is None:
        return (scorer, bodypart, coord)
    return (scorer, individual, bodypart, coord)


def has_bodypart(bp):
    return bp in bodyparts


def get_point(row, bodypart):
    x = row[col_key(bodypart, "x")]
    y = row[col_key(bodypart, "y")]
    p = row[col_key(bodypart, "likelihood")]
    return float(x), float(y), float(p)


def valid_point(pt, cutoff):
    x, y, p = pt
    return np.isfinite(x) and np.isfinite(y) and p >= cutoff


def find_dlc_index(image_path):
    candidates = [
        str(image_path),
        image_path.name,
        str(image_path.resolve())
    ]

    for c in candidates:
        if c in df.index:
            return c

    return None


def clamp_box(x1, y1, x2, y2, W, H):
    x1 = max(0, int(round(x1)))
    y1 = max(0, int(round(y1)))
    x2 = min(W, int(round(x2)))
    y2 = min(H, int(round(y2)))

    if x2 <= x1 or y2 <= y1:
        return None

    return x1, y1, x2, y2


def unit_vector(dx, dy):
    norm = np.sqrt(dx ** 2 + dy ** 2)
    if norm < 1e-6:
        return 0.0, 1.0, 1.0
    return dx / norm, dy / norm, norm


# =========================
# Safety check
# =========================
for bp in [NOSE, EYE_L, EYE_R]:
    if not has_bodypart(bp):
        raise ValueError(f"Missing bodypart: {bp}. Available: {bodyparts}")


# =========================
# Main loop
# =========================
saved = 0
skipped_not_in_dlc = 0
skipped_low_confidence = 0
skipped_bad_box = 0

for image_path in tqdm(sorted(image_dir.glob("*"))):
    if image_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
        continue

    dlc_index = find_dlc_index(image_path)

    if dlc_index is None:
        skipped_not_in_dlc += 1
        continue

    row = df.loc[dlc_index]

    nose = get_point(row, NOSE)
    eye_l = get_point(row, EYE_L)
    eye_r = get_point(row, EYE_R)

    if not valid_point(nose, PCUTOFF_MAIN):
        skipped_low_confidence += 1
        continue

    nose_x, nose_y, _ = nose

    eye_l_ok = valid_point(eye_l, PCUTOFF_MAIN)
    eye_r_ok = valid_point(eye_r, PCUTOFF_MAIN)

    if not eye_l_ok and not eye_r_ok:
        skipped_low_confidence += 1
        continue

    img = Image.open(image_path).convert("RGB")
    W, H = img.size

    optional_points = []
    for bp in OPTIONAL_PARTS:
        if has_bodypart(bp):
            pt = get_point(row, bp)
            if valid_point(pt, PCUTOFF_OPTIONAL):
                optional_points.append(pt)

    # =========================
    # FRONT VIEW
    # 两只眼睛都可靠
    # =========================
    if eye_l_ok and eye_r_ok:
        eye_l_x, eye_l_y, _ = eye_l
        eye_r_x, eye_r_y, _ = eye_r

        eye_mid_x = (eye_l_x + eye_r_x) / 2
        eye_mid_y = (eye_l_y + eye_r_y) / 2

        eye_dist = np.sqrt(
            (eye_l_x - eye_r_x) ** 2 +
            (eye_l_y - eye_r_y) ** 2
        )

        scale = max(eye_dist, 1.0)

        # 从 eye midpoint 指向 nose
        vx, vy, _ = unit_vector(nose_x - eye_mid_x, nose_y - eye_mid_y)

        # crop 中心放在 nose 再往下/嘴巴方向一点
        center_x = nose_x + vx * scale * NOSE_FORWARD_SHIFT
        center_y = nose_y + vy * scale * NOSE_FORWARD_SHIFT

        # front view 额外向图像下方移动一点
        center_y += scale * DOWN_SHIFT_FRONT

        crop_w = max(scale * FRONT_W_SCALE, MIN_W)
        crop_h = max(scale * FRONT_H_SCALE, MIN_H)

        x1 = center_x - crop_w / 2
        x2 = center_x + crop_w / 2
        y1 = center_y - crop_h / 2
        y2 = center_y + crop_h / 2

        # 强制上边界低于眼睛附近，避免把眼睛裁进去
        eye_low_y = max(eye_l_y, eye_r_y)
        min_y1 = eye_low_y + scale * REMOVE_EYE_MARGIN
        y1 = max(y1, min_y1)

    # =========================
    # SIDE VIEW
    # 只有一只眼睛可靠
    # =========================
    else:
        eye = eye_l if eye_l_ok else eye_r
        eye_x, eye_y, _ = eye

        # 从可见眼睛指向 nose
        vx, vy, eye_nose_dist = unit_vector(nose_x - eye_x, nose_y - eye_y)

        scale = max(eye_nose_dist, 1.0)

        # side view: crop 中心在 nose 前方
        center_x = nose_x + vx * scale * FORWARD_SHIFT_SIDE
        center_y = nose_y + vy * scale * FORWARD_SHIFT_SIDE

        crop_w = max(scale * SIDE_W_SCALE, MIN_W)
        crop_h = max(scale * SIDE_H_SCALE, MIN_H)

        x1 = center_x - crop_w / 2
        x2 = center_x + crop_w / 2
        y1 = center_y - crop_h / 2
        y2 = center_y + crop_h / 2

        # 上边界不要包含眼睛
        min_y1 = eye_y + scale * REMOVE_EYE_MARGIN
        if nose_y > eye_y:
            y1 = max(y1, min_y1)

    # =========================
    # 强制包含 nose
    # =========================
    nose_margin_x = 25
    nose_margin_y = 20

    x1 = min(x1, nose_x - nose_margin_x)
    x2 = max(x2, nose_x + nose_margin_x)
    y1 = min(y1, nose_y - nose_margin_y)
    y2 = max(y2, nose_y + nose_margin_y)

    # =========================
    # 如果有 mouth / whisker 点，也包含进去
    # =========================
    for x, y, p in optional_points:
        margin = 15
        x1 = min(x1, x - margin)
        x2 = max(x2, x + margin)
        y1 = min(y1, y - margin)
        y2 = max(y2, y + margin)

    box = clamp_box(x1, y1, x2, y2, W, H)

    if box is None:
        skipped_bad_box += 1
        continue

    x1, y1, x2, y2 = box

    crop = img.crop((x1, y1, x2, y2))
    crop = crop.resize((OUTPUT_SIZE, OUTPUT_SIZE))

    crop.save(output_dir / image_path.name)
    saved += 1


print("Saved crops:", saved)
print("Skipped because not in DLC index:", skipped_not_in_dlc)
print("Skipped because low confidence:", skipped_low_confidence)
print("Skipped because bad crop box:", skipped_bad_box)
print("Output directory:", output_dir)

In [ ]:
print(output_dir)

In [ ]:
import matplotlib.pyplot as plt
import random

crop_paths = list(output_dir.glob("*"))

sample_paths = random.sample(crop_paths, min(25, len(crop_paths)))

plt.figure(figsize=(12, 12))

for i, path in enumerate(sample_paths):
    img = Image.open(path)
    plt.subplot(5, 5, i + 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(path.name[:10])

plt.tight_layout()
plt.show()